In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pyspark import SparkContext, SparkConf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [43]:
import pandas as pd
import os

In [2]:
# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis")
sc = SparkContext(conf=conf)

In [3]:
pwd

'/content'

In [4]:
BASE_PATH = "/content/drive/MyDrive/ds200/assignment-3/"

In [46]:
def save_txt(name, data_rdd):
    results = data_rdd.collect()
    with open(f"{BASE_PATH}{name}.txt", "w", encoding="utf-8") as f:
        for item in results:
            if isinstance(item, (tuple, list)):
                line = ", ".join(map(str, item))
            else:
                line = str(item)
            f.write(line + "\n")
    print(f"Đã lưu {BASE_PATH}{name}.txt")

# Bài 1

In [5]:
# --- Bước 1: Đọc file movies.txt và tạo một map (MovieID -> Title)
movies_rdd = sc.textFile(BASE_PATH + "movies.txt")
# Tách dòng bằng dấu phẩy và lấy MovieID (index 0) làm key, Title (index 1) làm value[cite: 2]
movie_titles = movies_rdd.map(lambda line: line.split(",")) \
                         .map(lambda x: (x[0], x[1]))

In [6]:
movies_rdd.take(5)

['1001,The Godfather (1972),Crime|Drama',
 '1002,The Shawshank Redemption (1994),Drama',
 "1003,Schindler's List (1993),Biography|Drama|History",
 '1004,Raging Bull (1980),Biography|Drama|Sport',
 '1005,Casablanca (1942),Drama|Romance|War']

In [7]:
# --- Bước 2: Đọc file ratings_1.txt và ratings_2.txt, map MovieID -> (Rating, 1) ---
# Schema: UserID, MovieID, Rating, Timestamp
ratings_rdd = sc.textFile(BASE_PATH + "ratings_1.txt" + "," + BASE_PATH + "ratings_2.txt")
# Tách dòng, lấy MovieID (index 1) làm key, cặp (Rating, 1) làm value để tính tổng và đếm sau này[cite: 4, 5]
movie_ratings_mapped = ratings_rdd.map(lambda line: line.split(",")) \
                                  .map(lambda x: (x[1], (float(x[2]), 1)))

In [8]:
ratings_rdd.take(5)

['7,1020,4.5,1577836800',
 '23,1015,3.5,1577923200',
 '45,1030,4.0,1578009600',
 '12,1047,3.0,1578096000',
 '38,1012,4.5,1578182400']

In [9]:
movie_ratings_mapped.take(5)

[('1020', (4.5, 1)),
 ('1015', (3.5, 1)),
 ('1030', (4.0, 1)),
 ('1047', (3.0, 1)),
 ('1012', (4.5, 1))]

In [10]:
# --- Bước 3: Reduce để tính tổng điểm và số lượt đánh giá ---
movie_totals = movie_ratings_mapped.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

In [11]:
# Số lượt đánh giá, trung bình điểm của mỗi phim:
movie_summary = movie_totals.mapValues(lambda x: (x[0] / x[1], x[1]))

final_summary = movie_summary.join(movie_titles).map(lambda x: (x[0], x[1][1], x[1][0][0], x[1][0][1]))

In [12]:
# --- Bước 4: Tính điểm trung bình, lọc ra phim có ít nhất 5 lượt đánh giá ---
# 5 lượt đánh giá[cite: 1].
top_movies = movie_summary.filter(lambda x: x[1][1] >= 5)

In [13]:
# --- Bước 5: Tìm phim có điểm trung bình cao nhất ---
top_stats = top_movies.join(movie_titles) \
                         .map(lambda x: (x[0], x[1][1], x[1][0][0], x[1][0][1]))
                         # Kết quả: (MovieID, Title, AvgRating, Count)

In [14]:
# Tìm phim có AvgRating cao nhất bằng hàm takeOrdered (sort desc theo AvgRating)
top1_movie = top_stats.takeOrdered(1, key=lambda x: -x[2])

In [15]:
print("Danh sách phim và điểm trung bình của phim, số lượt đánh giá của phim:")
for movie in final_summary.collect():
  print(f"ID: {movie[0]} | Tên: {movie[1]} | Điểm TB: {movie[2]:.2f} | Tổng lượt: {movie[3]}")


print("Danh sách phim thỏa điều kiện >= 5 lượt đánh giá:")
for movie in top_stats.collect():
    print(f"ID: {movie[0]} | Tên: {movie[1]} | Điểm TB: {movie[2]:.2f} | Tổng lượt: {movie[3]}")

if top1_movie:
    m = top1_movie[0]
    print(f"\n=> Phim có điểm trung bình cao nhất: {m[1]} (ID: {m[0]}) với {m[2]:.2f} điểm.")

Danh sách phim và điểm trung bình của phim, số lượt đánh giá của phim:
ID: 1040 | Tên: Gladiator (2000) | Điểm TB: 3.61 | Tổng lượt: 18
ID: 1025 | Tên: The Terminator (1984) | Điểm TB: 4.06 | Tổng lượt: 18
ID: 1028 | Tên: Fight Club (1999) | Điểm TB: 3.50 | Tổng lượt: 7
ID: 1020 | Tên: E.T. the Extra-Terrestrial (1982) | Điểm TB: 3.67 | Tổng lượt: 18
ID: 1015 | Tên: Sunset Boulevard (1950) | Điểm TB: 4.36 | Tổng lượt: 7
ID: 1047 | Tên: The Social Network (2010) | Điểm TB: 3.86 | Tổng lượt: 7
ID: 1012 | Tên: Psycho (1960) | Điểm TB: 4.00 | Tổng lượt: 2
ID: 1050 | Tên: Mad Max: Fury Road (2015) | Điểm TB: 3.47 | Tổng lượt: 18
ID: 1037 | Tên: The Lord of the Rings: The Fellowship of the Ring (2001) | Điểm TB: 3.89 | Tổng lượt: 18
ID: 1013 | Tên: The Godfather: Part II (1974) | Điểm TB: 4.00 | Tổng lượt: 17
ID: 1039 | Tên: The Lord of the Rings: The Return of the King (2003) | Điểm TB: 3.82 | Tổng lượt: 11
ID: 1030 | Tên: The Silence of the Lambs (1991) | Điểm TB: 3.14 | Tổng lượt: 7
ID: 1

In [47]:
save_txt("23521330_Bai1", final_summary)

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai1.txt


# Bài 2

In [16]:
# Bước 1: Tạo map (MovieID -> List of Genres)
movie_genres_rdd = movies_rdd.map(lambda line: line.split(",")) \
                             .map(lambda x: (x[0], x[2].split("|")))

In [17]:
# Bước 2: Map từ MovieID -> Rating -> (Genre, Rating)
movie_ratings = ratings_rdd.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[1], float(x[2])))

In [18]:
joined_genre_ratings = movie_genres_rdd.join(movie_ratings)

In [19]:
genre_rating_pairs = joined_genre_ratings.flatMap(lambda x: [(genre, (x[1][1], 1)) for genre in x[1][0]])

In [20]:
# Bước 3:
genre_totals = genre_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
genre_averages = genre_totals.mapValues(lambda x: x[0] / x[1])

In [21]:
print("Điểm trung bình theo từng thể loại phim:")
for genre, avg in genre_averages.sortBy(lambda x: x[1], ascending=False).collect():
    print(f"Thể loại: {genre} | Điểm TB: {avg:.2f}")

Điểm trung bình theo từng thể loại phim:
Thể loại: Film-Noir | Điểm TB: 4.36
Thể loại: Horror | Điểm TB: 4.00
Thể loại: Mystery | Điểm TB: 4.00
Thể loại: Fantasy | Điểm TB: 3.86
Thể loại: Crime | Điểm TB: 3.81
Thể loại: Drama | Điểm TB: 3.76
Thể loại: Sci-Fi | Điểm TB: 3.73
Thể loại: Action | Điểm TB: 3.71
Thể loại: Thriller | Điểm TB: 3.70
Thể loại: Family | Điểm TB: 3.67
Thể loại: Adventure | Điểm TB: 3.63
Thể loại: Biography | Điểm TB: 3.56


In [48]:
save_txt("23521330_Bai2", genre_averages)

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai2.txt


# Bài 3

In [22]:
users_rdd = sc.textFile(BASE_PATH + "users.txt")

# Bước 1: tạo map
user_gender_map = users_rdd.map(lambda line: line.split(",")) \
                           .map(lambda x: (x[0], x[1]))

In [23]:
user_ratings = ratings_rdd.map(lambda line: line.split(",")) \
                          .map(lambda x: (x[0], (x[1], float(x[2]))))

In [24]:
joined_user_ratings = user_ratings.join(user_gender_map)

In [25]:
movie_gender_pairs = joined_user_ratings.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))

In [26]:
movie_gender_totals = movie_gender_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

movie_gender_averages = movie_gender_totals.mapValues(lambda x: x[0] / x[1])

In [27]:
results_with_names = movie_gender_averages.map(lambda x: (x[0][0], (x[0][1], x[1]))) \
                                          .join(movie_titles)

In [28]:
print(f"{'Tên phim'} | {'Giới tính'} | {'Điểm TB'}")
print("-" * 65)
for movie_id, ((gender, avg), title) in results_with_names.take(10):
    gender_label = "Nữ (F)" if gender == 'F' else "Nam (M)"
    print(f"{title} | {gender_label} | {avg:.2f}")

Tên phim | Giới tính | Điểm TB
-----------------------------------------------------------------
The Lord of the Rings: The Fellowship of the Ring (2001) | Nam (M) | 4.00
The Lord of the Rings: The Fellowship of the Ring (2001) | Nữ (F) | 3.80
Mad Max: Fury Road (2015) | Nữ (F) | 3.32
Mad Max: Fury Road (2015) | Nam (M) | 4.00
Sunset Boulevard (1950) | Nam (M) | 4.33
Sunset Boulevard (1950) | Nữ (F) | 4.50
No Country for Old Men (2007) | Nam (M) | 3.92
No Country for Old Men (2007) | Nữ (F) | 3.83
Fight Club (1999) | Nữ (F) | 3.50
Fight Club (1999) | Nam (M) | 3.50


In [49]:
save_txt("23521330_Bai3", results_with_names.map(lambda x: (x[1][1], x[1][0][0], f"{x[1][0][1]:.2f}")))

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai3.txt


# Bài 4

In [29]:
def get_age_group(age):
    age = int(age)
    if age < 18: return "Under 18"
    if age <= 24: return "18-24"
    if age <= 34: return "25-34"
    if age <= 44: return "35-44"
    if age <= 49: return "45-49"
    if age <= 55: return "50-55"
    return "56+"

In [30]:
user_age_map = users_rdd.map(lambda line: line.split(",")) \
                        .map(lambda x: (x[0], get_age_group(x[2])))

In [31]:
joined_age_ratings = user_ratings.join(user_age_map)

# Key: (MovieID, AgeGroup), Value: (Rating, 1)
movie_age_pairs = joined_age_ratings.map(lambda x: ((x[1][0][0], x[1][1]), (x[1][0][1], 1)))

In [32]:
movie_age_totals = movie_age_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

In [33]:
movie_age_averages = movie_age_totals.mapValues(lambda x: x[0] / x[1])

In [34]:
print(f"{'MovieID'} | {'Nhóm tuổi'} | {'Điểm TB'}")
for ((m_id, age_g), avg) in movie_age_averages.take(5):
    print(f"{m_id} | {age_g} | {avg:.2f}")

MovieID | Nhóm tuổi | Điểm TB
1025 | 50-55 | 4.00
1012 | 25-34 | 4.50
1050 | 25-34 | 3.40
1039 | 25-34 | 3.83
1040 | 45-49 | 4.00


In [50]:
save_txt("23521330_Bai4", movie_age_averages.map(lambda x: (x[0][0], x[0][1], f"{x[1]:.2f}")))

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai4.txt


# Bài 5

In [35]:
occ_names_rdd = sc.textFile(BASE_PATH + "occupation.txt") \
                  .map(lambda line: line.split(",")) \
                  .map(lambda x: (x[0], x[1]))
user_occ_map = users_rdd.map(lambda line: line.split(",")) \
                        .map(lambda x: (x[0], x[3]))

In [36]:
joined_occ_ratings = user_ratings.join(user_occ_map)
occ_rating_pairs = joined_occ_ratings.map(lambda x: (x[1][1], (x[1][0][1], 1)))
occ_totals = occ_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

occ_averages = occ_totals.mapValues(lambda x: (x[0] / x[1], x[1])) \
                         .join(occ_names_rdd)

In [37]:
final_occ_results = occ_averages.map(lambda x: (x[1][1], x[1][0][0], x[1][0][1]))

In [38]:
print(f"{'Nghề nghiệp':} | {'Điểm TB':} | {'Tổng lượt':}")
print("-" * 40)
for job, avg, count in final_occ_results.sortBy(lambda x: x[1], ascending=False).collect():
    print(f"{job} | {avg:.2f}  | {count}")

Nghề nghiệp | Điểm TB | Tổng lượt
----------------------------------------
Programmer | 4.25  | 10
Designer | 4.00  | 13
Student | 4.00  | 8
Nurse | 3.86  | 11
Consultant | 3.86  | 14
Journalist | 3.85  | 17
Artist | 3.73  | 11
Teacher | 3.70  | 5
Doctor | 3.69  | 21
Lawyer | 3.65  | 17
Salesperson | 3.65  | 17
Accountant | 3.58  | 6
Engineer | 3.56  | 18
Manager | 3.47  | 16


In [51]:
save_txt("23521330_Bai5", final_occ_results)

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai5.txt


# Bài 6

In [39]:
import datetime

In [40]:
def extract_year(timestamp):
    return datetime.datetime.fromtimestamp(int(timestamp)).year

In [41]:
year_rating_pairs = ratings_rdd.map(lambda line: line.split(",")) \
                               .map(lambda x: (extract_year(x[3]), (float(x[2]), 1)))
year_totals = year_rating_pairs.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
year_analysis = year_totals.mapValues(lambda x: (x[0] / x[1], x[1]))

In [42]:
print(f"{'Năm':<10} | {'Điểm TB':<10} | {'Tổng lượt đánh giá':<15}")
print("-" * 45)
for year, stats in year_analysis.sortByKey().collect():
    print(f"{year} | {stats[0]:.2f} | {stats[1]}")

Năm        | Điểm TB    | Tổng lượt đánh giá
---------------------------------------------
2020 | 3.75 | 184


In [52]:
save_txt("23521330_Bai6", year_analysis.map(lambda x: (x[0], f"{x[1][0]:.2f}", x[1][1])))

Đã lưu /content/drive/MyDrive/ds200/assignment-3/23521330_Bai6.txt
